In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Resizing
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input 

model = Sequential([
   
    Input(shape=(32, 32, 3)), 
    Resizing(96, 96),
    
    EfficientNetB0(include_top=False, weights='imagenet'),
    
])

# for layout warrnings
tf.config.optimizer.set_experimental_options({'layout_optimizer': False})

# 1. Load Data
(train_X, train_Y), (test_X, test_Y) = cifar10.load_data()

# NOTE: EfficientNet natively expects pixel values from 0-255! 
# We do NOT divide by 255.0 here like we did for our custom CNN.
train_Y = to_categorical(train_Y)
test_Y = to_categorical(test_Y)

# 2. Build the Transfer Learning Architecture
print("Downloading Pre-Trained EfficientNet Brain...")
model = Sequential([
    # Step A: Resize the tiny 32x32 CIFAR images to 96x96 so EfficientNet can "see" them
    Resizing(96, 96, input_shape=(32, 32, 3)),
    
    # Step B: Insert the pre-trained EfficientNet (without its original ImageNet head)
    EfficientNetB0(include_top=False, weights='imagenet', input_shape=(96, 96, 3)),
    
    # Step C: Add our custom classification head for CIFAR-10
    GlobalAveragePooling2D(),
    Dropout(0.5), # High dropout to prevent memorization
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

# 3. Two-Step Training Process

# Phase 1: Freeze the base model (Only train our new custom head)
model.layers[1].trainable = False 
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- PHASE 1: Training the custom head ---")
history_1 = model.fit(train_X, train_Y, 
                      validation_data=(test_X, test_Y), 
                      epochs=5, 
                      batch_size=64)

# Phase 2: Unfreeze the whole model for "Fine Tuning"
print("\n--- PHASE 2: Fine-Tuning the entire network ---")
model.layers[1].trainable = True 

# Use a VERY small learning rate so we don't wreck the pre-trained weights
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history_2 = model.fit(train_X, train_Y, 
                      validation_data=(test_X, test_Y), 
                      epochs=15, 
                      batch_size=64,
                      callbacks=callbacks)

# 4. Final Evaluation
test_loss, test_acc = model.evaluate(test_X, test_Y, verbose=0)
print(f"Transfer Learning Final Accuracy: {test_acc * 100:.2f}%")



--- PHASE 1: Training the custom head ---
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 66s 53ms/step - accuracy: 0.7739 - loss: 0.6715 - val_accuracy: 0.8482 - val_loss: 0.4501
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.8130 - loss: 0.5464 - val_accuracy: 0.8565 - val_loss: 0.4277
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.8225 - loss: 0.5151 - val_accuracy: 0.8554 - val_loss: 0.4173
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.8284 - loss: 0.4971 - val_accuracy: 0.8578 - val_loss: 0.4149
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.8343 - loss: 0.4792 - val_accuracy: 0.8607 - val_loss: 0.3998

--- PHASE 2: Fine-Tuning the entire network ---
Epoch 1/15
781/782 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5688 - loss: 1.7953

2026-08-25 06:08:49.398815: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:08:49.532038: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:08:52.719201: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:08:52.855379: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:08:53.044240: E external/local_xla/xla/stream_

782/782 ━━━━━━━━━━━━━━━━━━━━ 193s 141ms/step - accuracy: 0.6228 - loss: 1.4837 - val_accuracy: 0.7885 - val_loss: 0.6762 - learning_rate: 1.0000e-05
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 48s 61ms/step - accuracy: 0.7338 - loss: 0.9430 - val_accuracy: 0.8258 - val_loss: 0.5389 - learning_rate: 1.0000e-05
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 60ms/step - accuracy: 0.7779 - loss: 0.7446 - val_accuracy: 0.8519 - val_loss: 0.4598 - learning_rate: 1.0000e-05
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 48s 61ms/step - accuracy: 0.8088 - loss: 0.6283 - val_accuracy: 0.8704 - val_loss: 0.4062 - learning_rate: 1.0000e-05
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 60ms/step - accuracy: 0.8284 - loss: 0.5474 - val_accuracy: 0.8812 - val_loss: 0.3692 - learning_rate: 1.0000e-05
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.8435 - loss: 0.4881 - val_accuracy: 0.8891 - val_loss: 0.3389 - learning_rate: 1.0000e-05
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 60ms/step - accur

2026-08-25 06:20:53.883572: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:20:54.017303: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:20:54.355372: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:20:54.497958: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-25 06:20:55.368409: E external/local_xla/xla/stream_


Transfer Learning Final Accuracy: 92.47%
